# ⚡ JavaScript 기초 (JavaScript Basic) — VSCode 실습용

---

변수, 자료형, 문자열, 배열, 객체, 조건문, 반복문, 함수, 클래스, 비동기 처리, 최신 ES2024-2025 기능까지 포괄

> **실행 환경**: VSCode + Python 커널 + Node.js (로컬 환경)
> **표준**: ECMAScript 2025 (ES16) 기준
> **참고**: HTML, CSS는 별도 노트북에서 다룸

## 0. VSCode 환경 설정

### 0-1. Node.js 설치

JavaScript를 실행하려면 **Node.js**가 필요함

1. https://nodejs.org 에서 **LTS 버전** 다운로드 및 설치
2. 설치 후 터미널에서 확인:
```bash
node --version    # 예: v24.x.x
npm --version     # 예: 10.x.x
```

### 0-2. 가상환경 생성 및 활성화

```bash
# 1) .venv 가상환경 생성 (최초 1회)
python -m venv .venv

# 2) 가상환경 활성화
# Windows:
.venv\Scripts\activate
# Mac/Linux:
source .venv/bin/activate

# 3) 필수 패키지 설치
pip install ipykernel
```

### 0-3. VSCode 확장(Extension) 설치

| 확장 | 용도 |
|------|------|
| **Python** (Microsoft) | Python 커널 실행 |
| **Jupyter** (Microsoft) | .ipynb 파일 실행 |

### 0-4. 커널(Kernel) 선택

노트북 파일을 열면 **커널을 선택**해야 셀을 실행할 수 있음

1. `.ipynb` 파일 열기
2. 우측 상단 **"커널 선택(Select Kernel)"** 클릭
3. **"Python 환경(Python Environments)"** → `.venv` 가상환경 선택

> ⚠️ 커널을 선택하지 않으면 셀 실행 시 에러가 발생합니다.
> 커널이 목록에 보이지 않으면 `pip install ipykernel`을 다시 실행한 뒤, VSCode를 재시작하세요.

### 0-5. VSCode 인터프리터 설정

`Ctrl+Shift+P` → **"Python: Select Interpreter"** → `.venv` 경로 선택

> **이 노트북의 실행 방식**: Python 커널에서 `subprocess`로 Node.js를 호출하여 JavaScript 실행
> → 각 셀의 JavaScript 코드가 독립된 Node.js 프로세스에서 실행됨
> → 셀 간 변수 공유는 되지 않음 (각 셀이 독립 실행)

## 1. JavaScript란?

**JavaScript**: 웹페이지에 **동적 기능**을 추가하는 프로그래밍 언어

| 구분 | 역할 | 비유 |
|------|------|------|
| **HTML** | 웹페이지의 구조(뼈대) | 건물의 골조 |
| **CSS** | 웹페이지의 디자인(외관) | 건물의 인테리어 |
| **JavaScript** | 웹페이지의 **동작**(상호작용) | 건물의 전기·설비 |

### JavaScript의 특징

| 특징 | 설명 |
|------|------|
| **인터프리터 언어** | 컴파일 없이 코드를 한 줄씩 실행 |
| **동적 타입** | 변수의 자료형이 실행 중에 자동 결정 (Python과 유사) |
| **멀티 패러다임** | 객체지향, 함수형, 이벤트 기반 프로그래밍 모두 지원 |
| **실행 환경** | 브라우저(프론트엔드) + Node.js(백엔드) 모두 실행 가능 |

### ECMAScript란?

- JavaScript의 **공식 표준 명세**
- TC39 위원회가 매년 6월 새 버전 발표
- ES6(2015)에서 대규모 업데이트 → 이후 매년 소규모 업데이트
- **현재**: ECMAScript 2025 (ES16)

In [ ]:
# ──────────────────────────────────────────────
# JavaScript 실행 헬퍼 함수 정의 + Node.js 확인
# ──────────────────────────────────────────────
# 이 셀을 반드시 먼저 실행해야 이후 셀에서 JavaScript 코드를 실행할 수 있음
import subprocess
import tempfile
import os

def run_js(code):
    """Node.js로 JavaScript 코드를 실행하고 결과를 출력하는 헬퍼 함수"""
    tmp_path = os.path.join(tempfile.gettempdir(), '_jupyter_js_temp.js')
    with open(tmp_path, 'w', encoding='utf-8') as f:
        f.write(code)
    try:
        result = subprocess.run(
            ['node', tmp_path],
            capture_output=True, text=True, timeout=30
        )
        if result.stdout:
            print(result.stdout, end='')
        if result.stderr:
            print("[Error]", result.stderr.strip())
    except subprocess.TimeoutExpired:
        print("[Error] 실행 시간 초과 (30초)")
    except FileNotFoundError:
        print("[Error] Node.js가 설치되지 않았습니다. https://nodejs.org 에서 설치하세요.")
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass

# Node.js 버전 확인
result = subprocess.run(['node', '--version'], capture_output=True, text=True)
print(f"Node.js 버전: {result.stdout.strip()}")

# 테스트 실행
run_js('console.log("JavaScript 실행 테스트 성공!");')

## 2. 변수와 상수 (let, const, var)

**변수(variable)**: 데이터를 저장하는 이름 붙은 공간

| 키워드 | 재할당 | 재선언 | 스코프 | 권장 여부 |
|--------|--------|--------|--------|----------|
| `const` | 불가 | 불가 | 블록 `{}` | **기본으로 사용** |
| `let` | 가능 | 불가 | 블록 `{}` | 재할당 필요할 때 |
| `var` | 가능 | 가능 | 함수 | 사용 금지 (레거시) |

> **모범 사례**: 기본적으로 `const`를 사용하고, 값이 변경되어야 할 때만 `let` 사용
> `var`는 호이스팅, 스코프 문제가 있으므로 절대 사용하지 않음

In [ ]:
# ──────────────────────────────────────────────
# 변수 선언: const, let, var
# ──────────────────────────────────────────────
run_js("""
// === const: 상수 (재할당 불가) — 기본으로 사용 ===
const PI = 3.14159;
const name = "홍길동";
console.log("PI:", PI);           // 3.14159
console.log("name:", name);       // "홍길동"
// PI = 3.14;  // TypeError: Assignment to constant variable.

// const로 선언한 객체/배열은 내부 값 변경은 가능 (참조 자체만 고정)
const arr = [1, 2, 3];
arr.push(4);              // 내부 값 추가는 가능
console.log("arr:", arr); // [1, 2, 3, 4]
// arr = [5, 6];          // TypeError: 참조 자체 변경은 불가

// === let: 변수 (재할당 가능) — 값이 바뀔 때 사용 ===
let score = 80;
console.log("score:", score);     // 80
score = 95;                       // 재할당 가능
console.log("score 변경:", score); // 95

// === var: 사용 금지 (레거시) — 호이스팅 문제 ===
// var는 블록 스코프가 아닌 함수 스코프이므로 예상치 못한 버그 발생
console.log("var 호이스팅:", x);  // undefined (에러가 아님! — 위험)
var x = 10;
// let이나 const로 같은 코드를 작성하면 ReferenceError 발생 (안전)
""")

## 3. 자료형 (Data Types)

JavaScript의 자료형은 **원시 타입(Primitive)**과 **참조 타입(Reference)**으로 나뉨

### 원시 타입 (7가지) — 값 자체를 저장

| 타입 | 설명 | 예시 |
|------|------|------|
| `string` | 문자열 | `"안녕"`, `'hello'`, `` `템플릿` `` |
| `number` | 숫자 (정수 + 실수 통합) | `42`, `3.14`, `NaN`, `Infinity` |
| `boolean` | 참/거짓 | `true`, `false` |
| `null` | 의도적으로 비어있음 | `null` |
| `undefined` | 값이 할당되지 않음 | `undefined` |
| `symbol` | 고유한 식별자 (ES6+) | `Symbol("id")` |
| `bigint` | 큰 정수 (ES2020+) | `9007199254740993n` |

### 참조 타입 — 메모리 주소(참조)를 저장

| 타입 | 설명 | 예시 |
|------|------|------|
| `object` | 키-값 쌍의 컬렉션 | `{ name: "홍길동" }` |
| `array` | 순서가 있는 리스트 (객체의 일종) | `[1, 2, 3]` |
| `function` | 실행 가능한 코드 블록 (객체의 일종) | `function() {}` |

> **`typeof` 연산자**: 값의 자료형을 문자열로 반환

In [ ]:
# ──────────────────────────────────────────────
# 자료형 확인: typeof 연산자
# ──────────────────────────────────────────────
run_js("""
// 원시 타입 확인
console.log(typeof "안녕하세요");     // "string"
console.log(typeof 42);              // "number"
console.log(typeof 3.14);            // "number" (정수/실수 구분 없음)
console.log(typeof true);            // "boolean"
console.log(typeof null);            // "object" (역사적 버그, 실제로는 null)
console.log(typeof undefined);       // "undefined"
console.log(typeof Symbol("id"));    // "symbol"
console.log(typeof 123n);            // "bigint"

console.log("--- 참조 타입 ---");
console.log(typeof { a: 1 });        // "object"
console.log(typeof [1, 2, 3]);       // "object" (배열도 객체)
console.log(typeof function(){});    // "function"

// 배열 여부 확인: Array.isArray()
console.log(Array.isArray([1, 2]));  // true
console.log(Array.isArray({a: 1}));  // false
""")

## 4. 문자열 (String)

텍스트 데이터를 다루는 자료형. 3가지 방식으로 생성 가능

| 방식 | 문법 | 특징 |
|------|------|------|
| 큰따옴표 | `"문자열"` | 가장 일반적 |
| 작은따옴표 | `'문자열'` | 큰따옴표와 동일 |
| 백틱(템플릿 리터럴) | `` `문자열` `` | 변수 삽입, 여러 줄 지원 (ES6+, 권장) |

In [ ]:
# ──────────────────────────────────────────────
# 문자열 생성과 기본 조작
# ──────────────────────────────────────────────
run_js("""
// 문자열 생성 3가지 방식
const s1 = "큰따옴표 문자열";
const s2 = '작은따옴표 문자열';
const s3 = `백틱 문자열 (템플릿 리터럴)`;
console.log(s1, s2, s3);

// 문자열 연결 (concatenation)
const first = "Hello";
const second = "World";
console.log(first + " " + second);  // "Hello World"

// 문자열 길이
console.log("안녕하세요".length);    // 5

// 인덱싱 (0부터 시작)
const text = "JavaScript";
console.log(text[0]);      // "J"   — 첫 번째 글자
console.log(text.at(-1));  // "t"   — 마지막 글자 (at: ES2022+, 음수 인덱스 지원)

// 문자열은 불변(immutable) — 개별 문자 변경 불가
// text[0] = "j";  // 에러는 아니지만 변경되지 않음
""")

### 4-1. 문자열 주요 메서드

In [ ]:
# ──────────────────────────────────────────────
# 문자열 메서드 (검색, 변환, 추출)
# ──────────────────────────────────────────────
run_js("""
const s = "Hello, JavaScript World!";

// === 검색 ===
console.log(s.includes("Java"));       // true  — 포함 여부
console.log(s.startsWith("Hello"));    // true  — 시작 여부
console.log(s.endsWith("!"));          // true  — 끝 여부
console.log(s.indexOf("Java"));        // 7     — 위치 (없으면 -1)

// === 변환 ===
console.log(s.toUpperCase());          // "HELLO, JAVASCRIPT WORLD!"
console.log(s.toLowerCase());          // "hello, javascript world!"
console.log("  공백 제거  ".trim());   // "공백 제거"
console.log(s.replace("World", "세계"));    // 첫 번째만 대체
console.log(s.replaceAll("l", "L"));        // 모든 l → L (ES2021+)

// === 추출 ===
console.log(s.slice(7, 17));           // "JavaScript" — [시작, 끝) 추출
console.log(s.slice(-6));              // "orld!" — 음수: 뒤에서부터
console.log(s.split(", "));           // ["Hello", "JavaScript World!"] — 분리 → 배열

// === 반복, 채우기 ===
console.log("Ha".repeat(3));           // "HaHaHa"
console.log("5".padStart(3, "0"));     // "005" — 왼쪽 채우기
console.log("5".padEnd(3, "0"));       // "500" — 오른쪽 채우기
""")

### 4-2. 템플릿 리터럴 (Template Literal)

백틱(`` ` ``)으로 감싸는 문자열. `${ }` 안에 **변수나 표현식을 직접 삽입** 가능

```javascript
`텍스트 ${변수} 텍스트 ${표현식}`
```

In [ ]:
# ──────────────────────────────────────────────
# 템플릿 리터럴 — 변수 삽입, 여러 줄, 표현식
# ──────────────────────────────────────────────
run_js("""
const name = "BTS";
const members = 7;

// 변수 삽입 (기존 + 연결 방식 vs 템플릿 리터럴)
console.log(name + "는 " + members + "인조 그룹");    // 기존 방식
console.log(`${name}는 ${members}인조 그룹`);          // 템플릿 리터럴 (권장)

// 표현식 삽입
console.log(`2 + 3 = ${2 + 3}`);                       // "2 + 3 = 5"
console.log(`대문자: ${name.toLowerCase()}`);            // "대문자: bts"

// 여러 줄 문자열 (줄바꿈 그대로 유지)
const poem = `어느 날 세상이 멈췄어
아무런 예고도 하나 없이
봄은 기다림을 몰라서
눈치 없이 와버렸어`;
console.log(poem);
""")

## 5. 숫자와 연산자 (Number & Operators)

JavaScript는 정수와 실수를 **하나의 `number` 타입**으로 처리 (64비트 부동소수점)

### 산술 연산자

| 연산자 | 설명 | 예시 |
|--------|------|------|
| `+` | 더하기 | `5 + 3` → `8` |
| `-` | 빼기 | `5 - 3` → `2` |
| `*` | 곱하기 | `5 * 3` → `15` |
| `/` | 나누기 | `5 / 3` → `1.6667` |
| `%` | 나머지 | `7 % 3` → `1` |
| `**` | 거듭제곱 | `2 ** 3` → `8` |

### 특수 숫자 값

| 값 | 설명 |
|-----|------|
| `NaN` | Not a Number (숫자가 아님) |
| `Infinity` | 무한대 |
| `-Infinity` | 음의 무한대 |

In [ ]:
# ──────────────────────────────────────────────
# 숫자와 산술 연산
# ──────────────────────────────────────────────
run_js("""
// 기본 산술 연산
console.log("5 + 3 =", 5 + 3);       // 8
console.log("5 - 3 =", 5 - 3);       // 2
console.log("5 * 3 =", 5 * 3);       // 15
console.log("5 / 3 =", (5 / 3).toFixed(2));  // "1.67" (소수점 2자리)
console.log("7 % 3 =", 7 % 3);       // 1 (나머지)
console.log("2 ** 10 =", 2 ** 10);   // 1024 (거듭제곱)

// 증감 연산자
let x = 10;
x++;          // x = x + 1
console.log("x++ :", x);   // 11
x--;          // x = x - 1
console.log("x-- :", x);   // 10

// 복합 할당 연산자
x += 5;   // x = x + 5 → 15
x *= 2;   // x = x * 2 → 30
console.log("복합 할당 후:", x);

// 특수 값
console.log("NaN:", typeof NaN, Number.isNaN(NaN));   // "number", true
console.log("Infinity:", 1 / 0);                       // Infinity
console.log("문자+숫자:", "5" + 3);   // "53" (문자열 연결! 주의!)
console.log("문자-숫자:", "5" - 3);   // 2 (자동 숫자 변환)

// 숫자 변환
console.log(Number("42"));       // 42
console.log(parseInt("3.14"));   // 3 (정수 변환)
console.log(parseFloat("3.14")); // 3.14 (실수 변환)

// Math 내장 객체
console.log("Math.round(3.6):", Math.round(3.6));   // 4 (반올림)
console.log("Math.floor(3.9):", Math.floor(3.9));   // 3 (내림)
console.log("Math.ceil(3.1):", Math.ceil(3.1));     // 4 (올림)
console.log("Math.max(1,5,3):", Math.max(1, 5, 3)); // 5
console.log("Math.random():", Math.random().toFixed(4)); // 0~1 랜덤
""")

## 6. 불리언, 비교 연산자, 논리 연산자

### 비교 연산자

| 연산자 | 설명 | 예시 |
|--------|------|------|
| `===` | **일치** (값 + 타입 모두 비교) | `5 === "5"` → `false` |
| `!==` | **불일치** (값 + 타입 모두 비교) | `5 !== "5"` → `true` |
| `==` | 동등 (타입 자동 변환 후 비교, 비권장) | `5 == "5"` → `true` |
| `>`, `>=`, `<`, `<=` | 크기 비교 | `5 > 3` → `true` |

> **`===` vs `==`**: 항상 `===` (일치) 사용을 권장. `==`는 암묵적 타입 변환으로 예상치 못한 결과 발생

### 논리 연산자

| 연산자 | 설명 | 예시 |
|--------|------|------|
| `&&` | AND (둘 다 참) | `true && false` → `false` |
| `\|\|` | OR (하나라도 참) | `true \|\| false` → `true` |
| `!` | NOT (반대) | `!true` → `false` |

In [ ]:
# ──────────────────────────────────────────────
# 비교 연산자와 논리 연산자
# ──────────────────────────────────────────────
run_js("""
// === vs == (일치 vs 동등)
console.log("5 === 5:", 5 === 5);         // true  (값 + 타입 일치)
console.log('5 === "5":', 5 === "5");     // false (타입 다름)
console.log('5 == "5":', 5 == "5");       // true  (타입 변환 후 비교 — 위험!)
console.log("null == undefined:", null == undefined);   // true (둘 다 '비어있음')
console.log("null === undefined:", null === undefined); // false (타입 다름)

// 논리 연산자
console.log("true && false:", true && false);     // false
console.log("true || false:", true || false);     // true
console.log("!true:", !true);                     // false

// Falsy 값: false로 평가되는 값들 (조건문에서 중요)
// false, 0, "", null, undefined, NaN → 모두 falsy
console.log("--- Falsy 값 ---");
console.log(Boolean(false));     // false
console.log(Boolean(0));         // false
console.log(Boolean(""));        // false (빈 문자열)
console.log(Boolean(null));      // false
console.log(Boolean(undefined)); // false
console.log(Boolean(NaN));       // false

// Truthy: 위 6개를 제외한 모든 값
console.log(Boolean("hello"));   // true
console.log(Boolean(42));        // true
console.log(Boolean([]));        // true (빈 배열도 truthy!)
console.log(Boolean({}));        // true (빈 객체도 truthy!)
""")

### 6-1. null, undefined, 그리고 Nullish 처리

| 값 | 의미 | 발생 상황 |
|----|------|----------|
| `null` | 의도적으로 "비어있음" | 개발자가 직접 할당 |
| `undefined` | 값이 아직 할당되지 않음 | 변수 선언만 하고 할당 안 한 경우 |

### 옵셔널 체이닝(`?.`)과 Nullish 병합(`??`)

| 연산자 | 설명 | 예시 |
|--------|------|------|
| `?.` | 값이 null/undefined이면 undefined 반환 (에러 방지) | `obj?.name` |
| `??` | 왼쪽이 null/undefined일 때만 오른쪽 값 사용 | `null ?? "기본값"` |
| `??=` | null/undefined일 때만 할당 | `x ??= 10` |

In [ ]:
# ──────────────────────────────────────────────
# null, undefined, 옵셔널 체이닝, Nullish 병합
# ──────────────────────────────────────────────
run_js("""
// null vs undefined
let a;                    // 선언만 → undefined
const b = null;           // 의도적 빈 값 → null
console.log("a:", a);     // undefined
console.log("b:", b);     // null

// 옵셔널 체이닝 (?.) — null/undefined에서 속성 접근 시 에러 방지
const user = { name: "홍길동", address: { city: "서울" } };
console.log(user.address?.city);      // "서울"
console.log(user.phone?.number);      // undefined (에러 아님!)
// console.log(user.phone.number);    // TypeError 발생!

// Nullish 병합 (??) — null/undefined일 때만 기본값 사용
console.log(null ?? "기본값");       // "기본값"
console.log(undefined ?? "기본값");  // "기본값"
console.log(0 ?? "기본값");          // 0 (0은 null/undefined 아님)
console.log("" ?? "기본값");         // "" (빈 문자열도 null 아님)

// || 와의 차이: ||는 falsy 값 전체를 대체
console.log(0 || "기본값");          // "기본값" (0이 falsy라서 대체됨)
console.log(0 ?? "기본값");          // 0 (??는 null/undefined만 대체)

// Nullish 할당 (??=)
let config = { timeout: null, retries: 3 };
config.timeout ??= 5000;   // null이므로 5000 할당
config.retries ??= 10;     // 3이므로 변경 안 됨
console.log(config);        // { timeout: 5000, retries: 3 }
""")

## 7. 배열 (Array)

여러 개의 값을 **순서대로** 저장하는 자료구조 (Python의 리스트와 유사)
- **대괄호(`[ ]`)**로 생성
- 인덱스는 **0부터** 시작
- 서로 다른 타입의 값을 혼합하여 저장 가능

In [ ]:
# ──────────────────────────────────────────────
# 배열 생성, 인덱싱, 기본 조작
# ──────────────────────────────────────────────
run_js("""
// 배열 생성
const fruits = ["사과", "바나나", "포도", "딸기", "수박"];
const numbers = [10, 20, 30, 40, 50];
const mixed = ["문자", 42, true, null, [1, 2]];  // 혼합 가능

// 길이
console.log("길이:", fruits.length);   // 5

// 인덱싱 (0부터 시작)
console.log("첫 번째:", fruits[0]);      // "사과"
console.log("마지막:", fruits.at(-1));   // "수박" (at: ES2022+)

// 요소 변경
fruits[1] = "망고";
console.log("변경 후:", fruits);   // ["사과", "망고", "포도", "딸기", "수박"]

// 포함 여부 확인
console.log("포도 포함?:", fruits.includes("포도"));   // true
console.log("위치:", fruits.indexOf("포도"));           // 2
""")

### 7-1. 배열 주요 메서드

In [ ]:
# ──────────────────────────────────────────────
# 배열 메서드: 추가, 삭제, 변환
# ──────────────────────────────────────────────
run_js("""
// === 추가/삭제 (원본 변경) ===
const arr = ["a", "b", "c"];

arr.push("d");          // 맨 뒤에 추가
console.log("push:", arr);         // ["a", "b", "c", "d"]

arr.pop();              // 맨 뒤에서 제거
console.log("pop:", arr);          // ["a", "b", "c"]

arr.unshift("z");       // 맨 앞에 추가
console.log("unshift:", arr);      // ["z", "a", "b", "c"]

arr.shift();            // 맨 앞에서 제거
console.log("shift:", arr);        // ["a", "b", "c"]

arr.splice(1, 1, "B", "B2");  // 인덱스 1에서 1개 삭제 후 "B", "B2" 삽입
console.log("splice:", arr);       // ["a", "B", "B2", "c"]

// === 변환 (원본 유지, 새 배열 반환) ===
const nums = [3, 1, 4, 1, 5, 9];
console.log("slice(1,4):", nums.slice(1, 4));   // [1, 4, 1] — 추출
console.log("concat:", nums.concat([2, 6]));     // [3,1,4,1,5,9,2,6] — 연결
console.log("join:", nums.join("-"));             // "3-1-4-1-5-9" — 문자열 합치기
console.log("flat:", [1, [2, [3]]].flat(2));     // [1, 2, 3] — 중첩 배열 펼치기

// === 정렬 ===
const names = ["다", "가", "나"];
console.log("sort:", [...names].sort());          // ["가", "나", "다"]
console.log("reverse:", [...names].reverse());    // ["나", "가", "다"]
// [...names] — 스프레드로 복사 후 정렬 (원본 보존)

// 숫자 정렬 주의: sort()는 기본적으로 문자열 비교
console.log([10, 9, 2, 100].sort());              // [10, 100, 2, 9] (잘못됨!)
console.log([10, 9, 2, 100].sort((a, b) => a - b)); // [2, 9, 10, 100] (올바름)
""")

### 7-2. 배열 고급 메서드 (map, filter, reduce, find)

배열의 각 요소에 **함수를 적용**하여 변환하는 메서드 — JavaScript에서 매우 자주 사용

| 메서드 | 반환값 | 설명 |
|--------|--------|------|
| `map(fn)` | 새 배열 | 각 요소를 변환한 결과 배열 |
| `filter(fn)` | 새 배열 | 조건을 만족하는 요소만 추출 |
| `reduce(fn, init)` | 단일 값 | 모든 요소를 하나로 누적 |
| `find(fn)` | 요소 1개 | 조건을 만족하는 **첫 번째** 요소 |
| `findIndex(fn)` | 인덱스 | 조건을 만족하는 첫 요소의 위치 |
| `some(fn)` | boolean | 하나라도 조건 만족하면 true |
| `every(fn)` | boolean | 모두 조건 만족하면 true |
| `forEach(fn)` | undefined | 각 요소에 함수 실행 (반환값 없음) |

In [ ]:
# ──────────────────────────────────────────────
# 배열 고급 메서드: map, filter, reduce, find
# ──────────────────────────────────────────────
run_js("""
const numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10];

// map: 각 요소를 변환하여 새 배열 반환
const doubled = numbers.map(n => n * 2);
console.log("map (x2):", doubled);   // [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]

// filter: 조건을 만족하는 요소만 추출
const evens = numbers.filter(n => n % 2 === 0);
console.log("filter (짝수):", evens);   // [2, 4, 6, 8, 10]

// reduce: 모든 요소를 하나의 값으로 누적
const sum = numbers.reduce((acc, cur) => acc + cur, 0);
console.log("reduce (합계):", sum);   // 55
// acc: 누적값, cur: 현재 요소, 0: 초기값

// find: 조건을 만족하는 첫 번째 요소
const found = numbers.find(n => n > 5);
console.log("find (>5):", found);   // 6

// findIndex: 조건을 만족하는 첫 요소의 인덱스
console.log("findIndex (>5):", numbers.findIndex(n => n > 5));   // 5

// some / every: 조건 만족 여부
console.log("some (>8):", numbers.some(n => n > 8));     // true
console.log("every (>0):", numbers.every(n => n > 0));   // true

// 메서드 체이닝: map + filter + reduce 연결
const result = numbers
    .filter(n => n % 2 === 0)    // 짝수만: [2, 4, 6, 8, 10]
    .map(n => n ** 2)            // 제곱: [4, 16, 36, 64, 100]
    .reduce((a, b) => a + b, 0); // 합계: 220
console.log("체이닝 결과:", result);
""")

## 8. 객체 (Object)

**키(key)와 값(value)의 쌍**으로 데이터를 저장하는 자료구조 (Python의 딕셔너리와 유사)

```javascript
const 객체 = {
    키1: 값1,
    키2: 값2,
    메서드: function() { ... }
};
```

In [ ]:
# ──────────────────────────────────────────────
# 객체 생성, 접근, 수정
# ──────────────────────────────────────────────
run_js("""
// 객체 생성
const person = {
    name: "홍길동",
    age: 25,
    city: "서울",
    hobbies: ["독서", "코딩"],
    greet() {   // 메서드 (축약형)
        return `안녕하세요, ${this.name}입니다.`;
    }
};

// 속성 접근: 점 표기법 vs 괄호 표기법
console.log(person.name);           // "홍길동" (점 표기법)
console.log(person["city"]);        // "서울" (괄호 표기법)
console.log(person.greet());        // "안녕하세요, 홍길동입니다."

// 속성 추가/수정/삭제
person.email = "hong@mail.com";     // 추가
person.age = 26;                    // 수정
delete person.email;                // 삭제
console.log(person);

// 속성 존재 확인
console.log("name" in person);                     // true
console.log(person.hasOwnProperty("phone"));        // false

// Object 주요 메서드
console.log("keys:", Object.keys(person));         // 키 배열
console.log("values:", Object.values(person));     // 값 배열
console.log("entries:", Object.entries(person));    // [키, 값] 쌍 배열
""")

## 9. 구조분해 할당과 스프레드 연산자

### 구조분해 할당 (Destructuring)

배열이나 객체의 값을 **개별 변수로 한 번에 추출**하는 문법 (ES6+)

### 스프레드 연산자 (`...`)

배열이나 객체를 **펼쳐서** 개별 요소로 분리 (ES6+)

### 나머지 연산자 (`...`)

나머지 요소들을 **하나로 모음** (스프레드와 동일 기호, 위치에 따라 역할 다름)

In [ ]:
# ──────────────────────────────────────────────
# 구조분해 할당 + 스프레드/나머지 연산자
# ──────────────────────────────────────────────
run_js("""
// === 배열 구조분해 ===
const [a, b, c] = [10, 20, 30];
console.log(a, b, c);   // 10 20 30

const [first, ...rest] = [1, 2, 3, 4, 5];  // 나머지 연산자
console.log("first:", first);   // 1
console.log("rest:", rest);     // [2, 3, 4, 5]

const [x = 0, y = 0, z = 0] = [10, 20];  // 기본값 설정
console.log(x, y, z);   // 10 20 0

// === 객체 구조분해 ===
const person = { name: "홍길동", age: 25, city: "서울" };
const { name, age, city } = person;
console.log(name, age, city);   // "홍길동" 25 "서울"

const { name: userName, age: userAge } = person;  // 다른 변수명으로 할당
console.log(userName, userAge);   // "홍길동" 25

const { country = "한국" } = person;  // 기본값 설정
console.log(country);   // "한국" (person에 없으므로 기본값)

// === 스프레드 연산자 (...) ===
// 배열 스프레드
const arr1 = [1, 2, 3];
const arr2 = [4, 5, 6];
const combined = [...arr1, ...arr2];
console.log("배열 합치기:", combined);   // [1, 2, 3, 4, 5, 6]

const arrCopy = [...arr1];              // 배열 복사
console.log("배열 복사:", arrCopy);

// 객체 스프레드
const defaults = { theme: "dark", lang: "ko", size: 14 };
const userConfig = { lang: "en", size: 16 };
const config = { ...defaults, ...userConfig };  // 뒤의 값이 덮어씀
console.log("객체 병합:", config);   // { theme:"dark", lang:"en", size:16 }
""")

## 10. 조건문 (Conditional Statements)

조건을 판단하여 **해당 조건에 맞는 코드만 실행**

### if - else if - else

```javascript
if (조건1) {
    // 조건1이 true일 때
} else if (조건2) {
    // 조건2가 true일 때
} else {
    // 모든 조건이 false일 때
}
```

### 삼항 연산자 (Ternary Operator)

```javascript
조건 ? 참일_때_값 : 거짓일_때_값
```

### switch 문

```javascript
switch (값) {
    case 값1: 실행문; break;
    case 값2: 실행문; break;
    default: 기본 실행문;
}
```

In [ ]:
# ──────────────────────────────────────────────
# 조건문: if / else if / else, 삼항 연산자, switch
# ──────────────────────────────────────────────
run_js("""
// === if - else if - else ===
const score = 85;

if (score >= 90) {
    console.log("A등급");
} else if (score >= 80) {
    console.log("B등급");
} else if (score >= 70) {
    console.log("C등급");
} else {
    console.log("F등급");
}
// 출력: "B등급"

// === 삼항 연산자 (간단한 조건에 유용) ===
const age = 20;
const status = age >= 18 ? "성인" : "미성년자";
console.log(`${age}세: ${status}`);   // "20세: 성인"

// === switch 문 (여러 값 중 하나를 선택) ===
const day = new Date().getDay();  // 0(일) ~ 6(토)
switch (day) {
    case 0: console.log("일요일"); break;
    case 1: console.log("월요일"); break;
    case 2: console.log("화요일"); break;
    case 3: console.log("수요일"); break;
    case 4: console.log("목요일"); break;
    case 5: console.log("금요일"); break;
    case 6: console.log("토요일"); break;
    default: console.log("알 수 없음");
}
// break가 없으면 아래 case도 연달아 실행됨 (fall-through 주의)
""")

## 11. 반복문 (Loops)

코드를 **반복 실행**하는 구문

| 종류 | 용도 |
|------|------|
| `for` | 횟수 기반 반복 |
| `while` | 조건 기반 반복 |
| `for...of` | 배열/이터러블 순회 (값) |
| `for...in` | 객체 속성 순회 (키) |

In [ ]:
# ──────────────────────────────────────────────
# 반복문: for, while
# ──────────────────────────────────────────────
run_js("""
// === for 문 (가장 기본) ===
// for (초기값; 조건; 증감) { 실행문 }
console.log("=== for 문: 1~5 ===");
for (let i = 1; i <= 5; i++) {
    console.log(`  ${i}번째 반복`);
}

// === while 문 (조건 기반) ===
console.log("=== while 문: 구구단 3단 ===");
let j = 1;
while (j <= 9) {
    console.log(`  3 x ${j} = ${3 * j}`);
    j++;
}

// === break와 continue ===
console.log("=== break: 5에서 멈춤 ===");
for (let i = 1; i <= 10; i++) {
    if (i > 5) break;      // 반복 중단
    console.log(`  ${i}`);
}

console.log("=== continue: 짝수만 출력 ===");
for (let i = 1; i <= 10; i++) {
    if (i % 2 !== 0) continue;  // 홀수면 건너뜀
    console.log(`  ${i}`);
}
""")

In [ ]:
# ──────────────────────────────────────────────
# 반복문: for...of (값 순회), for...in (키 순회)
# ──────────────────────────────────────────────
run_js("""
// === for...of: 배열/이터러블의 "값"을 순회 ===
const fruits = ["사과", "바나나", "포도"];
console.log("=== for...of (배열 값 순회) ===");
for (const fruit of fruits) {
    console.log(`  ${fruit}`);
}

// 문자열도 순회 가능
console.log("=== for...of (문자열 순회) ===");
for (const char of "Hello") {
    process.stdout.write(char + " ");
}
console.log();

// === for...in: 객체의 "키"를 순회 ===
const person = { name: "홍길동", age: 25, city: "서울" };
console.log("=== for...in (객체 키 순회) ===");
for (const key in person) {
    console.log(`  ${key}: ${person[key]}`);
}

// entries()로 인덱스와 값을 함께 순회
console.log("=== entries() (인덱스 + 값) ===");
for (const [idx, fruit] of fruits.entries()) {
    console.log(`  ${idx}: ${fruit}`);
}
""")

## 12. 함수 (Function)

입력(매개변수)을 받아 **특정 작업을 수행**하고 결과를 반환하는 코드 블록

### 함수 선언 방식 3가지

| 방식 | 문법 | 호이스팅 |
|------|------|----------|
| 함수 선언문 | `function add(a, b) { }` | O (선언 전 호출 가능) |
| 함수 표현식 | `const add = function(a, b) { }` | X |
| 화살표 함수 | `const add = (a, b) => { }` | X |

> **화살표 함수(Arrow Function)**: ES6+에서 추가된 간결한 함수 문법. 현대 JS에서 가장 많이 사용

In [ ]:
# ──────────────────────────────────────────────
# 함수: 선언문, 표현식, 매개변수
# ──────────────────────────────────────────────
run_js("""
// === 함수 선언문 ===
function add(a, b) {
    return a + b;
}
console.log("add(3, 5):", add(3, 5));   // 8

// === 기본값 매개변수 (Default Parameters) ===
function greet(name, greeting = "안녕하세요") {
    return `${greeting}, ${name}님!`;
}
console.log(greet("홍길동"));              // "안녕하세요, 홍길동님!"
console.log(greet("홍길동", "반갑습니다")); // "반갑습니다, 홍길동님!"

// === 나머지 매개변수 (Rest Parameters) ===
function sum(...numbers) {
    return numbers.reduce((acc, cur) => acc + cur, 0);
}
console.log("sum(1,2,3,4,5):", sum(1, 2, 3, 4, 5));   // 15

// === 반환값이 없는 함수 ===
function printInfo(name, age) {
    console.log(`이름: ${name}, 나이: ${age}`);
    // return이 없으면 undefined 반환
}
printInfo("홍길동", 25);
""")

### 12-1. 화살표 함수 (Arrow Function)

In [ ]:
# ──────────────────────────────────────────────
# 화살표 함수와 콜백 함수
# ──────────────────────────────────────────────
run_js("""
// === 화살표 함수 기본 문법 ===
// function 키워드 대신 => 사용
const add = (a, b) => {
    return a + b;
};

// 본문이 한 줄이면 중괄호와 return 생략 가능
const multiply = (a, b) => a * b;

// 매개변수가 1개면 소괄호 생략 가능
const square = x => x ** 2;

// 매개변수가 없으면 빈 소괄호 필수
const getRandom = () => Math.random();

console.log("add:", add(3, 5));         // 8
console.log("multiply:", multiply(4, 6)); // 24
console.log("square:", square(7));       // 49

// === 콜백 함수: 다른 함수의 인수로 전달되는 함수 ===
const numbers = [3, 1, 4, 1, 5, 9];

// map에 화살표 함수를 콜백으로 전달
const doubled = numbers.map(n => n * 2);
console.log("콜백(map):", doubled);

// sort에 비교 함수를 콜백으로 전달
const sorted = [...numbers].sort((a, b) => a - b);
console.log("콜백(sort):", sorted);

// 자주 쓰는 패턴: 배열 메서드 + 화살표 함수
const people = [
    { name: "홍길동", age: 25 },
    { name: "김철수", age: 32 },
    { name: "이영희", age: 28 }
];
const names = people.map(p => p.name);
const adults = people.filter(p => p.age >= 30);
console.log("이름 목록:", names);     // ["홍길동", "김철수", "이영희"]
console.log("30세 이상:", adults);    // [{ name:"김철수", age:32 }]
""")

## 13. 클래스 (Class)

**객체를 생성하기 위한 설계도(템플릿)** (ES6+)

```javascript
class 클래스이름 {
    constructor(매개변수) {    // 생성자: 객체 생성 시 자동 호출
        this.속성 = 매개변수;   // this: 생성될 객체 자신을 가리킴
    }
    메서드() {                  // 메서드 정의
        return 결과;
    }
}
const 객체 = new 클래스이름(값);  // new 키워드로 객체 생성
```

In [ ]:
# ──────────────────────────────────────────────
# 클래스: 생성자, 메서드, 상속, 정적 메서드
# ──────────────────────────────────────────────
run_js("""
// === 기본 클래스 ===
class Calculator {
    constructor(n1, n2) {    // 생성자
        this.n1 = n1;
        this.n2 = n2;
    }
    add()  { return this.n1 + this.n2; }
    sub()  { return this.n1 - this.n2; }
    mul()  { return this.n1 * this.n2; }
    div()  { return this.n1 / this.n2; }
    info() {
        return `Calculator(${this.n1}, ${this.n2})`;
    }
}

const calc = new Calculator(10, 3);
console.log(calc.info());           // "Calculator(10, 3)"
console.log("add:", calc.add());    // 13
console.log("sub:", calc.sub());    // 7

// === 상속 (extends) ===
class ScientificCalc extends Calculator {
    constructor(n1, n2) {
        super(n1, n2);       // 부모 생성자 호출 (필수)
    }
    power() { return this.n1 ** this.n2; }
    sqrt()  { return Math.sqrt(this.n1); }
}

const sci = new ScientificCalc(2, 10);
console.log("power:", sci.power());   // 1024 (2^10)
console.log("add:", sci.add());       // 12 (부모 메서드 사용 가능)

// === 정적 메서드 (static) — 인스턴스 없이 클래스에서 직접 호출 ===
class MathHelper {
    static average(...nums) {
        return nums.reduce((a, b) => a + b, 0) / nums.length;
    }
}
console.log("평균:", MathHelper.average(80, 90, 100));  // 90

// === private 필드 (#) — 외부 접근 차단 ===
class User {
    #password;                      // private 필드
    constructor(name, password) {
        this.name = name;
        this.#password = password;
    }
    checkPassword(input) {
        return this.#password === input;
    }
}

const user = new User("홍길동", "1234");
console.log(user.name);               // "홍길동"
// console.log(user.#password);        // SyntaxError (외부 접근 불가)
console.log(user.checkPassword("1234")); // true
""")

## 14. 에러 처리 (Error Handling)

`try...catch...finally`로 에러를 **잡아서 처리** (프로그램이 멈추지 않음)

```javascript
try {
    // 에러가 발생할 수 있는 코드
} catch (error) {
    // 에러 발생 시 실행되는 코드
} finally {
    // 에러 여부와 관계없이 항상 실행
}
```

In [ ]:
# ──────────────────────────────────────────────
# 에러 처리: try / catch / finally, throw
# ──────────────────────────────────────────────
run_js("""
// === 기본 try-catch ===
try {
    const data = JSON.parse("잘못된 JSON");
} catch (error) {
    console.log("에러 발생:", error.message);
    console.log("에러 타입:", error.name);    // SyntaxError
}

// === finally: 항상 실행 ===
try {
    console.log("--- try 실행 ---");
    // throw new Error("의도적 에러");  // 주석 해제하면 에러 발생
} catch (error) {
    console.log("catch:", error.message);
} finally {
    console.log("finally: 항상 실행됨");
}

// === throw: 에러를 직접 발생시키기 ===
function divide(a, b) {
    if (b === 0) {
        throw new Error("0으로 나눌 수 없습니다", { cause: "division by zero" });
    }
    return a / b;
}

try {
    console.log("10 / 2 =", divide(10, 2));   // 5
    console.log("10 / 0 =", divide(10, 0));   // 에러 발생
} catch (error) {
    console.log("에러:", error.message);        // "0으로 나눌 수 없습니다"
    console.log("원인:", error.cause);           // "division by zero" (ES2022+)
}

// === 주요 에러 타입 ===
// SyntaxError  — 문법 오류 (JSON 파싱 실패 등)
// TypeError    — 타입 오류 (null.속성 접근 등)
// ReferenceError — 정의되지 않은 변수 참조
// RangeError   — 범위 초과 (배열 크기 음수 등)
""")

## 15. 비동기 처리 (Asynchronous)

JavaScript는 **싱글 스레드**이지만, **비동기 처리**로 시간이 걸리는 작업을 효율적으로 수행

| 방식 | 설명 | 시대 |
|------|------|------|
| 콜백(Callback) | 함수 안에 함수 전달 | 초기 |
| Promise | 비동기 결과를 나타내는 객체 | ES6 (2015) |
| async/await | Promise를 동기식처럼 작성 | ES2017 (**권장**) |

### Promise의 3가지 상태

```
pending (대기) → fulfilled (성공, resolve) 또는 rejected (실패, reject)
```

In [ ]:
# ──────────────────────────────────────────────
# Promise 기초
# ──────────────────────────────────────────────
run_js("""
// === Promise 생성 ===
const myPromise = new Promise((resolve, reject) => {
    const success = true;
    if (success) {
        resolve("작업 성공!");     // 성공 시 resolve 호출
    } else {
        reject("작업 실패...");    // 실패 시 reject 호출
    }
});

// === then / catch / finally 로 결과 처리 ===
myPromise
    .then(result => console.log("then:", result))     // "작업 성공!"
    .catch(error => console.log("catch:", error))
    .finally(() => console.log("finally: 항상 실행"));

// === 실전 예시: setTimeout을 Promise로 감싸기 ===
function delay(ms) {
    return new Promise(resolve => setTimeout(resolve, ms));
}

delay(100).then(() => {
    console.log("100ms 후 실행됨");
});

// === Promise 유틸리티 메서드 ===
const p1 = Promise.resolve("A");
const p2 = Promise.resolve("B");
const p3 = Promise.resolve("C");

// Promise.all: 모두 성공해야 결과 반환
Promise.all([p1, p2, p3]).then(results => {
    console.log("all:", results);   // ["A", "B", "C"]
});

// Promise.allSettled: 성공/실패 관계없이 모든 결과 반환
const p4 = Promise.reject("에러!");
Promise.allSettled([p1, p4]).then(results => {
    console.log("allSettled:", JSON.stringify(results));
});
""")

### 15-1. async / await (권장)

`async` 함수 안에서 `await`를 사용하면 **Promise를 동기식처럼** 읽기 쉽게 작성 가능

```javascript
async function 함수명() {
    const 결과 = await Promise객체;   // Promise가 완료될 때까지 대기
    return 결과;
}
```

In [ ]:
# ──────────────────────────────────────────────
# async / await — 비동기 코드를 동기식처럼 작성
# ──────────────────────────────────────────────
run_js("""
// 비동기 함수 시뮬레이션
function fetchUser(id) {
    return new Promise((resolve, reject) => {
        setTimeout(() => {
            if (id > 0) {
                resolve({ id, name: "홍길동", age: 25 });
            } else {
                reject(new Error("유효하지 않은 ID"));
            }
        }, 50);
    });
}

// === async/await 사용 ===
async function getUserInfo() {
    try {
        console.log("사용자 정보 요청 중...");
        const user = await fetchUser(1);   // Promise 완료까지 대기
        console.log("사용자:", user);

        // 에러 처리 테스트
        const invalid = await fetchUser(-1);
    } catch (error) {
        console.log("에러 처리:", error.message);
    }
}

// async 함수 실행
getUserInfo();

// === Promise.withResolvers() (ES2024) ===
// Promise의 resolve/reject를 외부에서 제어
const { promise, resolve } = Promise.withResolvers();
setTimeout(() => resolve("외부에서 resolve!"), 50);
promise.then(val => console.log("withResolvers:", val));
""")

## 16. 모듈 (Module)

코드를 **파일 단위로 분리**하여 재사용하는 시스템 (ES6+)

### ES Modules (ESM) — 현재 표준

```javascript
// math.js (내보내기)
export function add(a, b) { return a + b; }
export const PI = 3.14159;
export default class Calculator { ... }  // 기본 내보내기 (파일당 1개)

// main.js (가져오기)
import Calculator, { add, PI } from './math.js';
```

| 문법 | 설명 |
|------|------|
| `export` | 개별 내보내기 (여러 개 가능) |
| `export default` | 기본 내보내기 (파일당 1개) |
| `import { ... } from` | 개별 가져오기 |
| `import 이름 from` | 기본 가져오기 |
| `import * as 별칭 from` | 전체 가져오기 |

> **CommonJS** (`require`/`module.exports`): Node.js의 레거시 모듈 시스템. 새 코드에서는 ESM 사용 권장
> Node.js에서 ESM 사용: 파일 확장자를 `.mjs`로 하거나, `package.json`에 `"type": "module"` 추가

In [ ]:
# ──────────────────────────────────────────────
# 모듈 시스템 구조 예시 (소스코드 안내)
# ──────────────────────────────────────────────
# 모듈은 여러 파일로 분리해야 하므로 소스코드로 안내

module_example = """
// ==============================
// math.js (모듈 파일 — 내보내기)
// ==============================
// 개별 내보내기 (named export)
export function add(a, b) {
    return a + b;
}

export function multiply(a, b) {
    return a * b;
}

export const PI = 3.14159;

// 기본 내보내기 (default export) — 파일당 1개
export default class Calculator {
    constructor(n1, n2) {
        this.n1 = n1;
        this.n2 = n2;
    }
    sum() {
        return this.n1 + this.n2;
    }
}


// ==============================
// main.js (메인 파일 — 가져오기)
// ==============================
// 기본 + 개별 가져오기
import Calculator, { add, multiply, PI } from './math.js';

console.log(add(3, 5));              // 8
console.log(PI);                      // 3.14159

const calc = new Calculator(10, 20);
console.log(calc.sum());              // 30

// 전체 가져오기 (네임스페이스)
import * as MathUtil from './math.js';
console.log(MathUtil.multiply(4, 5)); // 20
"""

print("=== ES Modules 구조 예시 ===")
print(module_example)

## 17. 최신 JavaScript 기능 (ES2024-2025)

### ES2024 (ES15) 주요 기능

| 기능 | 설명 |
|------|------|
| `Object.groupBy()` | 배열을 그룹별로 분류 |
| `Promise.withResolvers()` | Promise의 resolve/reject를 외부에서 제어 |
| `String.isWellFormed()` | 유니코드 문자열 유효성 확인 |

### ES2025 (ES16) 주요 기능

| 기능 | 설명 |
|------|------|
| **Set 메서드** | `union`, `intersection`, `difference` 등 집합 연산 |
| **Iterator Helpers** | `.map()`, `.filter()`, `.take()` 등 이터레이터 메서드 |
| `Promise.try()` | 동기/비동기 함수를 안전하게 Promise로 래핑 |
| `RegExp.escape()` | 정규식 문자열 이스케이프 |

> **Node.js 버전 요구사항**: ES2025 기능(Set 메서드, Iterator Helpers 등)은 **Node.js 22 이상**에서 지원됩니다.
> `node --version`으로 버전을 확인하고, 22 미만이면 https://nodejs.org 에서 최신 LTS 버전을 설치하세요.
> 아래 셀 실행 시 에러가 발생하면 Node.js 버전을 먼저 업그레이드하세요.

In [ ]:
# ──────────────────────────────────────────────
# ES2024: Object.groupBy, structuredClone
# ──────────────────────────────────────────────
run_js("""
// === Object.groupBy() — 배열을 그룹별로 분류 (ES2024) ===
const people = [
    { name: "홍길동", dept: "개발" },
    { name: "김철수", dept: "기획" },
    { name: "이영희", dept: "개발" },
    { name: "박민수", dept: "기획" },
    { name: "정수진", dept: "디자인" }
];

const byDept = Object.groupBy(people, person => person.dept);
console.log("부서별 그룹:");
for (const [dept, members] of Object.entries(byDept)) {
    console.log(`  ${dept}: ${members.map(m => m.name).join(", ")}`);
}

// === structuredClone() — 깊은 복사 (ES2022) ===
const original = { a: 1, b: { c: 2, d: [3, 4] } };
const clone = structuredClone(original);
clone.b.c = 999;
console.log("원본:", JSON.stringify(original));   // b.c가 2 (원본 유지)
console.log("복사본:", JSON.stringify(clone));     // b.c가 999
""")

In [ ]:
# ──────────────────────────────────────────────
# ES2025: Set 메서드, Iterator Helpers
# ──────────────────────────────────────────────
run_js("""
// === Set 메서드 — 집합 연산 (ES2025) ===
const frontEnd = new Set(["HTML", "CSS", "JavaScript", "React"]);
const backEnd = new Set(["Python", "JavaScript", "Node.js", "SQL"]);

console.log("=== Set 집합 연산 ===");
console.log("합집합:", frontEnd.union(backEnd));
console.log("교집합:", frontEnd.intersection(backEnd));
console.log("차집합:", frontEnd.difference(backEnd));
console.log("대칭차집합:", frontEnd.symmetricDifference(backEnd));
console.log("부분집합?:", new Set(["HTML"]).isSubsetOf(frontEnd));

// === Iterator Helpers — 이터레이터 메서드 (ES2025) ===
// 배열의 .values()로 이터레이터를 얻은 뒤 체이닝
console.log("\n=== Iterator Helpers ===");
const nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10];
const result = nums.values()
    .filter(n => n % 2 === 0)    // 짝수만
    .map(n => n ** 2)            // 제곱
    .take(3)                     // 처음 3개만
    .toArray();                  // 배열로 변환
console.log("짝수 제곱 (3개):", result);  // [4, 16, 36]

// 무한 이터레이터와 함께 사용
function* naturals() {
    let i = 1;
    while (true) yield i++;
}
const first10Squares = naturals()
    .map(n => n * n)
    .take(10)
    .toArray();
console.log("자연수 제곱 (10개):", first10Squares);
""")

---

## JavaScript 기초 요약 (Quick Reference)

| 카테고리 | 핵심 문법 | 설명 |
|---------|----------|------|
| **변수** | `const`, `let` | 상수(기본), 변수(재할당 필요시) |
| **자료형** | `string`, `number`, `boolean`, `null`, `undefined`, `object`, `array` | 원시 7 + 참조 타입 |
| **문자열** | `` `${변수}` ``, `.includes()`, `.slice()`, `.split()` | 템플릿 리터럴 + 메서드 |
| **배열** | `.push()`, `.map()`, `.filter()`, `.reduce()`, `.find()` | 추가, 변환, 필터, 누적, 검색 |
| **객체** | `{ key: value }`, `Object.keys()`, `Object.entries()` | 키-값 저장, 순회 |
| **구조분해** | `const { a, b } = obj`, `const [x, y] = arr` | 값 추출 |
| **스프레드** | `[...arr]`, `{...obj}` | 복사, 병합, 펼치기 |
| **조건문** | `if/else`, `switch`, `조건 ? A : B` | 분기 처리 |
| **반복문** | `for`, `while`, `for...of`, `for...in` | 반복 실행 |
| **함수** | `const fn = (x) => x * 2` | 화살표 함수 (권장) |
| **클래스** | `class A extends B { constructor() { super(); } }` | 객체 설계도, 상속 |
| **에러** | `try { } catch (e) { } finally { }` | 에러 처리 |
| **비동기** | `async function() { await promise }` | 비동기 처리 (권장) |
| **모듈** | `import { fn } from './file.js'` | 코드 분리 |
| **Nullish** | `?.` (옵셔널 체이닝), `??` (Nullish 병합) | null/undefined 안전 처리 |
| **ES2024** | `Object.groupBy()`, `structuredClone()` | 그룹화, 깊은 복사 |
| **ES2025** | Set 메서드, Iterator Helpers | 집합 연산, 이터레이터 체이닝 |

### 핵심 원칙 요약

| 원칙 | 설명 |
|------|------|
| **const 우선** | 기본 `const`, 재할당 필요 시만 `let`, `var` 절대 금지 |
| **=== 사용** | 비교 시 항상 `===` / `!==` 사용 (`==` / `!=` 금지) |
| **화살표 함수** | 간결한 콜백에는 화살표 함수 사용 |
| **템플릿 리터럴** | 문자열 조합에는 `+` 대신 백틱(`` ` ``) 사용 |
| **구조분해 활용** | 객체/배열에서 값 추출 시 구조분해 사용 |
| **async/await** | 비동기 처리는 async/await 사용 (then 체인 대신) |
| **ESM 사용** | `import`/`export` 사용 (`require` 대신) |